<h1> 📘 Biofilter — Report: Entity Neighborhood Summary </h1>

Resolve a heterogeneous list of inputs (genes, diseases, pathways, proteins, chemicals, GO terms) into entities and return a 1-hop neighborhood summary, with neighbor counts and primary names grouped by entity type.

Engine-agnostic: works on PostgreSQL **and** SQLite. Fuzzy matching uses `rapidfuzz` client-side, no DB extension required.


### 1. Start Biofilter


In [ ]:
from biofilter import Biofilter

bf = Biofilter(debug_mode=False)


### 2. Mixed inputs with type hints

Type prefixes (`gene:`, `disease:`, `pathway:`, `protein:`, `chemical:`, `go:`) scope the resolution to the matching `EntityGroup`. This avoids cross-domain matches when the same string exists in multiple groups.


In [ ]:
items = [
    "gene:BRCA1",
    "disease:Alzheimer disease",
    "pathway:DNA repair",
    "APOE",  # no type hint — searched across all groups
]

df = bf.report.run(
    "entity_neighborhood_summary",
    items=items,
    match_mode="exact",
    aliases_top_n=10,
    neighbors_top_n_per_type=20,
    emit_not_found_rows=True,
)

print(f"Rows: {len(df)}")
df.head()


### 3. Recommended demo columns


In [ ]:
cols = [
    "Input Word",
    "Entity ID",
    "Entity Type",
    "Exact Match",
    "Matched Name",
    "Primary Alias",
    "Degree Total (1-hop)",
    "Degree By Type (1-hop)",
    "Resolve Status",
]
df[[c for c in cols if c in df.columns]]


### 4. `like` mode — substring matches

Useful when the input is a partial term and you want to find every entity whose alias contains it. Multiple aliases of the same entity collapse into a single row.


In [ ]:
df_like = bf.report.run(
    "entity_neighborhood_summary",
    items=["pathway:signaling", "disease:alzheimer"],
    match_mode="like",
    neighbors_top_n_per_type=10,
)

print(f"Rows: {len(df_like)}")
df_like[["Input Word", "Matched Name", "Exact Match", "Primary Alias", "Entity Type"]].head(20)


### 5. `fuzzy` mode — similarity matching

Uses `rapidfuzz` token-sort ratio for typo-tolerant matching. Lower the `similarity_threshold` (default 80) when inputs are short forms (e.g. `"alzheimer"` vs `"Alzheimer disease"`).

In [ ]:
df_fuzzy = bf.report.run(
    "entity_neighborhood_summary",
    items=["gene:BRCA1", "disease:alzheimers"],  # legacy / typo
    match_mode="fuzzy",
    similarity_threshold=70,
)

df_fuzzy[["Input Word", "Matched Name", "Resolve Score", "Primary Alias"]]


### 6. Inspect the per-type neighbor lists

Each `EntityGroup` in the database becomes a column on the output (`Genes`, `Pathways`, `Diseases`, etc.) with a JSON-encoded list of neighbor primary names. Useful for quickly seeing what an input touches.


In [ ]:
import json

row = df.iloc[0]  # first resolved entity
print(f"{row['Input Word']} → {row['Primary Alias']} ({row['Entity Type']})")
print(f"Total neighbors: {row['Degree Total (1-hop)']}")
print(f"By type: {row['Degree By Type (1-hop)']}")

for col in ("Genes", "Pathways", "Diseases", "Proteins"):
    if col in df.columns:
        neighbors = json.loads(row[col]) if row[col] else []
        if neighbors:
            print(f"\n{col} ({len(neighbors)}):")
            for n in neighbors[:5]:
                print(f"  - {n}")
